# p-adic EML — Exploration

The p-adic numbers $\mathbb{Z}_p$ extend the integers with an alternative
notion of "closeness": $x$ and $y$ are p-adically close if $p^k \mid (x-y)$
for large $k$. The p-adic norm is $|x|_p = p^{-v_p(x)}$.

The EML operator extends to p-adic arithmetic:
$$\text{eml}_p(x, y) = \exp_p(x) - \log_p(y)$$

with convergence within the disk $|x|_p < p^{-1/(p-1)}$.


In [ ]:
from monogate.padic import (
    PAdicNumber, padic_exp, padic_log, padic_eml,
    padic_fixed_points, valuation
)
import math

print("p-adic EML module loaded.")

## 1. p-adic Basics — Arithmetic and Valuation

In [ ]:
# 2-adic valuation examples
print("2-adic valuations:")
for n in [1, 2, 4, 8, 12, 16, 24, 100]:
    print(f"  v_2({n:3d}) = {valuation(n, 2)}")

print("\n3-adic valuations:")
for n in [1, 3, 9, 27, 18, 45]:
    print(f"  v_3({n:3d}) = {valuation(n, 3)}")

In [ ]:
# Construct p-adic numbers from integers
p, prec = 3, 8

x = PAdicNumber.from_int(6, p=p, precision=prec)
y = PAdicNumber.from_int(9, p=p, precision=prec)

print(f"x = 6 in 3-adic: {x}")
print(f"  digits: {x.digits}")
print(f"  v_3(6) = {x.val}")
print(f"  |6|_3 = {x.norm():.4f}")

print(f"\ny = 9 = 3^2 in 3-adic: {y}")
print(f"  v_3(9) = {y.val}")
print(f"  |9|_3 = {y.norm():.4f}")

# Note: |9|_3 = 3^{-2} = 1/9 — 9 is "small" in 3-adic!
print("\n3-adically, 9 is SMALLER than 6 (more divisible by 3)")

In [ ]:
# Arithmetic
p, prec = 5, 10
a = PAdicNumber.from_int(12, p=p, precision=prec)
b = PAdicNumber.from_int(13, p=p, precision=prec)

print(f"a = 12 (5-adic), b = 13 (5-adic)")
print(f"a + b = {(a + b).to_int()} (should be 25 mod 5^10 = 25)")
print(f"a * b = {(a * b).to_int()} (should be 156)")
print(f"-a = {(-a).to_int()} (5-adic complement)")
print(f"a + (-a) = {(a + (-a)).to_int()} mod 5^{prec}")

## 2. p-adic Exponential and Logarithm

In [ ]:
# p-adic exp: converges for v_p(x) >= 2 (p=2) or v_p(x) >= 1 (p odd)
print("=== p-adic Exponential ===")

for p in (3, 5, 7):
    prec = 10
    # Use x = p (v_p(p) = 1 >= 1 for odd p)
    x = PAdicNumber.from_int(p, p=p, precision=prec)
    exp_x = padic_exp(x)
    print(f"\np={p}: x = {p}")
    print(f"  v_{p}(x) = {x.val}  (need >= 1)")
    print(f"  exp_{p}(x) digits: {exp_x.digits[:6]}...")
    print(f"  exp_{p}(x) as int: {exp_x.to_int()}")
    # Verify: exp(0) = 1
    z = PAdicNumber.zero(p=p, precision=prec)
    exp_0 = padic_exp(z)
    print(f"  exp_{p}(0) = {exp_0.to_int()} (should be 1)")

In [ ]:
# 2-adic exp: requires v_2(x) >= 2
p, prec = 2, 10
x4 = PAdicNumber.from_int(4, p=p, precision=prec)   # v_2(4) = 2 ✓
x8 = PAdicNumber.from_int(8, p=p, precision=prec)   # v_2(8) = 3 ✓

exp_4 = padic_exp(x4)
exp_8 = padic_exp(x8)

print(f"2-adic exp(4):  {exp_4.to_int()} (digits: {exp_4.digits[:8]})")
print(f"2-adic exp(8):  {exp_8.to_int()} (digits: {exp_8.digits[:8]})")

# Verify exp(x) * exp(-x) = 1 in 2-adic
nx4 = -x4
try:
    exp_neg4 = padic_exp(nx4)
    product = exp_4 * exp_neg4
    print(f"2-adic exp(4) * exp(-4) = {product.to_int()} (should be 1 mod 2^{prec})")
except ValueError as e:
    print(f"Note: exp(-4) outside convergence radius in 2-adic: {e}")

In [ ]:
# p-adic log: converges for x ≡ 1 (mod p)
print("=== p-adic Logarithm ===")

for p in (3, 5, 7):
    prec = 10
    # x ≡ 1 (mod p): use x = 1 + p
    x_int = 1 + p
    x = PAdicNumber.from_int(x_int, p=p, precision=prec)
    log_x = padic_log(x)
    print(f"\np={p}: x = 1 + {p} = {x_int}")
    print(f"  x ≡ 1 (mod {p})? {x_int % p == 1}")
    print(f"  log_{p}({x_int}) digits: {log_x.digits[:6]}...")
    # Verify: log(1) = 0
    one = PAdicNumber.one(p=p, precision=prec)
    log_1 = padic_log(one)
    print(f"  log_{p}(1) = {log_1.to_int()} (should be 0)")

## 3. p-adic EML Operator

In [ ]:
print("=== p-adic EML ===")
print("eml_p(x, y) = exp_p(x) - log_p(y)")
print()

# Key property: eml(0, 1) = exp(0) - log(1) = 1 - 0 = 1
for p in (2, 3, 5, 7):
    prec = 8
    z = PAdicNumber.zero(p=p, precision=prec)
    one = PAdicNumber.one(p=p, precision=prec)
    result = padic_eml(z, one)
    print(f"eml_{p}(0, 1) = {result.to_int()} (should be 1)  {'✓' if result.digits[0] == 1 else '✗'}")

In [ ]:
# Compare p-adic EML with real EML
import math

print("Comparison: real EML vs p-adic EML")
print(f"{'x (real)':12s} {'real eml(x,1)':16s} {'3-adic input':15s} {'3-adic eml output (int)':22s}")
print("-" * 70)

p, prec = 3, 8
for n in [0, 3, 6, 9, 12]:
    # Real EML: eml(n, 1) = exp(n) - log(1) = exp(n)
    real_val = math.exp(n)
    
    # 3-adic: use n * p as input (to ensure convergence)
    x_int = n * p
    try:
        x = PAdicNumber.from_int(x_int % (p**prec), p=p, precision=prec)
        one = PAdicNumber.one(p=p, precision=prec)
        r = padic_eml(x, one)
        print(f"{n:12.1f} {real_val:16.4f} {x_int:15d} {r.to_int():22d}")
    except ValueError:
        print(f"{n:12.1f} {real_val:16.4f} {x_int:15d} {'(outside convergence)':22s}")

## 4. Fixed Points and Attractors

In [ ]:
# Search for p-adic fixed points of f(x) = eml_p(x, 1)
print("=== p-adic Fixed Points ===")
print("Fixed point: x such that eml_p(x, 1) ≈ x")
print()

for p in (3, 5, 7):
    fps = padic_fixed_points(depth=1, p=p, precision=6, n_candidates=30)
    print(f"p={p}: {len(fps)} fixed point(s) found")
    for fp in fps:
        print(f"  x = {fp.to_int()} (digits: {fp.digits[:4]}..., |x|_{p} = {fp.norm():.4f})")

In [ ]:
# Compare with real attractor
# Real fixed point of exp(x) = x is the phantom attractor λ_crit ≈ -W(-1) ≈ 0.5671
# This has no real solution; the p-adic analog may differ
print("Real EML attractor (phantom): λ_crit ≈ 0.5671...")
print("In p-adic, the attractor structure differs by prime.")
print()
print("Depth-2 fixed points (f(x) = eml_p(eml_p(x,x), eml_p(x,x))):")
for p in (3, 5):
    fps2 = padic_fixed_points(depth=2, p=p, precision=6, n_candidates=20)
    print(f"  p={p}: {len(fps2)} fixed point(s)")

## 5. Convergence Analysis

In [ ]:
# Show how p-adic norm varies with input
print("p-adic norms for inputs 1..20 (p=3):")
print(f"{'n':5s} {'v_3(n)':8s} {'|n|_3':8s} {'in conv. disk?':15s}")
print("-" * 42)

p = 3
for n in range(1, 21):
    v = valuation(n, p)
    norm = p ** (-v)
    # Convergence: need |x|_p < p^(-1/(p-1)) = 3^(-1/2) ≈ 0.577
    conv_radius = p ** (-1/(p-1))
    in_conv = norm < conv_radius
    print(f"{n:5d} {v:8d} {norm:8.4f} {str(in_conv):15s}")

## 6. Open Questions

1. **Kummer connection**: Does the p-adic EML series have a combinatorial
   interpretation via carry patterns (Kummer's theorem)?

2. **Teichmüller lift**: Do the Teichmüller representatives ($(p-1)$-th
   roots of unity in $\mathbb{Z}_p$) form fixed points of the p-adic EML?

3. **p-adic Euler identity**: For $p \equiv 1 \pmod 4$, a p-adic $i$ exists.
   Does $\exp_p(i\pi_p) + 1 = 0$ hold in some p-adic sense?

4. **Lambert W function**: The real phantom attractor involves $W(-1)$.
   Is there a p-adic Lambert W, and does it relate to the p-adic EML
   fixed points found above?

5. **Universality**: Does p-adic EML generate all p-adic analytic functions
   (analogous to the real universality theorem C1)?